[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [9]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [10]:
import torch
import torch.nn as nn
import math

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # pass  # W_q, W_k, W_v, W_o
        self.num_heads=num_heads
        self.d_k=d_model//num_heads
        self.W_q=nn.Linear(d_model,d_model)
        self.W_k=nn.Linear(d_model,d_model)
        self.W_v=nn.Linear(d_model,d_model)
        self.W_o=nn.Linear(d_model,d_model)

    def forward(self, x_q, x_kv):
        # pass  # Q from x_q, K/V from x_kv, no causal mask
        B,S_q,_=x_q.shape
        B,S_kv,_=x_kv.shape

        Q,K,V=self.W_q(x_q),self.W_k(x_kv),self.W_v(x_kv)
        Q=Q.view(B,S_q,self.num_heads,self.d_k).permute(0,2,1,3)
        K=K.view(B,S_kv,self.num_heads,self.d_k).permute(0,2,1,3)
        V=V.view(B,S_kv,self.num_heads,self.d_k).permute(0,2,1,3)

        scores=torch.matmul(Q,K.transpose(-1,-2))/math.sqrt(self.d_k)
        weights=torch.softmax(scores,dim=-1)
        attn=torch.matmul(weights,V)
        return self.W_o(attn.permute(0,2,1,3).contiguous().view(B,S_q,-1))


In [12]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.5ms)
  ✅ [2/4] Q and KV different lengths (0.9ms)
  ✅ [3/4] No causal mask — all KV affects all Q (31.2ms)
  ✅ [4/4] Gradient flow (17.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (51.4ms total)
  Progress saved. Run status() to see your dashboard.

